## Quiz 17

#### Madison Ward

In [2]:
import requests
import pandas as pa
from bs4 import BeautifulSoup

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

r = requests.get('https://en.wikipedia.org/wiki/List_of_highest_mountains_on_Earth', headers=headers)
html_contents = r.text
html_soup = BeautifulSoup(html_contents,"lxml")
tables = html_soup.find_all('table',class_="wikitable")

df1 = pa.read_html(str(tables))[0]
df1.head()

/tmp/ipykernel_1227/2324499821.py:14: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df1 = pa.read_html(str(tables))[0]


,Rank[i],Mountain name(s),Height (rounded)[ii],Prominence (rounded)[iii],Range,Coordinates[iv],Parent mountain [v],First ascent[vi],Country/ Region administered by[a],Image
0,1,Mount EverestSagarmathaChomolungma,"8,849 metres (29,032 ft) [c]","8,849 metres (29,032 ft)",Mahalangur Himalaya,27°59′17″N 86°55′30″E﻿ / ﻿27.9881°N 86.925°E,—N/a,1953,NepalChina,NaN
1,2,K2,"8,611 metres (28,251 ft)","4,020 metres (13,190 ft)",Baltoro Karakoram,35°52′53″N 76°30′48″E﻿ / ﻿35.88139°N 76.51333°E,Mount Everest,1954,PakistanChina[d][14],NaN
2,3,Kangchenjunga,"8,586 metres (28,169 ft)","3,922 metres (12,867 ft)",Kangchenjunga Himalaya,27°42′12″N 88°08′51″E﻿ / ﻿27.70333°N 88.14750°E *,Mount Everest,1955,NepalIndia,NaN
3,4,Lhotse,"8,516 metres (27,940 ft)","610 metres (2,000 ft)",Mahalangur Himalaya,27°57′42″N 86°55′59″E﻿ / ﻿27.96167°N 86.93306°E,Mount Everest,1956,ChinaNepal,NaN
4,5,Makalu,"8,485 metres (27,838 ft)","2,378 metres (7,802 ft)",Mahalangur Himalaya,27°53′23″N 87°05′20″E﻿ / ﻿27.88972°N 87.08889°E,Mount Everest,1955,NepalChina,NaN


In [3]:
import re

def clean_column(col):
    col = re.sub(r"\[.*?\]", "", col)
    col = re.sub(r"\(.*?\)", "", col)
    col = col.strip()
    col = col.lower().replace(" ", "_")
    return col

df1.columns = [clean_column(col) for col in df1.columns]

df1.columns

Index(['rank', 'mountain_name', 'height', 'prominence', 'range', 'coordinates',
       'parent_mountain', 'first_ascent', 'country/_region_administered_by',
       'image'],
      dtype='object')

In [4]:
df1.columns = [re.sub(r"\(.*?\)|\[.*?\]", "", c).strip().lower().replace(" ", "_") for c in df1.columns]

In [11]:
import re

def clean_country(s):
    s = re.sub(r"\[.*?\]", "", str(s))
    countries = re.findall(r"[A-Z][a-z]+", s)
    return ", ".join(countries)
country_col_name_original = 'Country/Region administered by[a]'
country_col_name_cleaned = 'country/_region_administered_by'

if country_col_name_cleaned in df1.columns:
    source_column = country_col_name_cleaned
elif country_col_name_original in df1.columns:
    source_column = country_col_name_original
else:
    raise ValueError("Country column not found")

df1["country"] = df1[source_column].apply(clean_country)

df1["country"].head()

,country
0,"Nepal, China"
1,"Pakistan, China"
2,"Nepal, India"
3,"China, Nepal"
4,"Nepal, China"


A way I could clean the column is by splitting the combined country names by their capital letters. After that, I could store the countries as a string or list so the data can be easily grouped and analyzed.